# What is a SimDrive Object?

A `SimDrive` object is the central simulation unit in FASTSim. It combines a
`Vehicle` (what the vehicle is) and a `Cycle` (what the vehicle does) into a
single simulation scenario.

Calling `sd.walk()` steps the vehicle through the drive cycle one time step at
a time, computing energy flows, achieved speed, and component states at each
step.


In [1]:
import fastsim

## Creating a SimDrive

`SimDrive` takes a `Vehicle` and a `Cycle` as arguments. Optionally, a
`SimParams` object can be passed to customize solver behavior (see
[](editing-sim-params.ipynb)).

Load a bundled vehicle and drive cycle, then construct the simulation object:


In [2]:
# Load a bundled vehicle
veh = fastsim.Vehicle.from_resource("2012_Ford_Fusion.yaml")

# Load a bundled drive cycle
cyc = fastsim.Cycle.from_resource("udds.csv")

# Combine into a simulation scenario
sd = fastsim.SimDrive(veh, cyc)
print(sd)


SimDrive { veh: Vehicle { name: "2012 Ford Fusion", doc: None, year: 2012, pt_type: ConventionalVehicle(ConventionalVehicle { fs: FuelStorage { pwr_out_max: 1000000.0 m^2 kg^1 s^-3, pwr_ramp_lag: 1.0 s^1, energy_capacity: 2124000000.0 m^2 kg^1 s^-2, specific_energy: None, mass: None }, fc: FuelConverter { thrml: None, mass: None, specific_pwr: None, pwr_out_max: 130500.0 m^2 kg^1 s^-3, pwr_out_max_init: 21750.0 m^2 kg^1 s^-3, pwr_ramp_lag: 6.0 s^1, eff_interp_from_pwr_out: Interp1D(Interp1D { data: InterpData { grid: [[0.0, 0.005, 0.015, 0.04, 0.06, 0.1, 0.14, 0.2, 0.4, 0.6, 0.8, 1.0], shape=[12], strides=[1], layout=CFcf (0xf), const ndim=1], values: [0.1, 0.12, 0.16, 0.22, 0.28, 0.33, 0.35, 0.36, 0.35, 0.34, 0.32, 0.3], shape=[12], strides=[1], layout=CFcf (0xf), const ndim=1 }, strategy: Linear(Linear), extrapolate: Error }), pwr_for_peak_eff: 26100.0 m^2 kg^1 s^-3, pwr_idle_fuel: 0.0 m^2 kg^1 s^-3, state: FuelConverterState { i: TrackedState(0, Fresh), pwr_out_max: TrackedState(0.0

## Running the Simulation

Call `sd.walk()` to execute the simulation. This steps through every time step
in the cycle and computes the vehicle's energy flows, achieved speed, and
component states.

`walk()` applies powertrain-specific corrections automatically:

- **Conventional**: simulates once.
- **BEV / PHEV**: sets initial SOC to the maximum before simulating.
- **HEV**: iterates until the initial and final SOC are balanced (charge-sustaining operation).

If you need to run exactly one iteration without any of those corrections (e.g.,
you have already set an initial SOC yourself in the RES `state`), use `sd.walk_once()` instead.


In [3]:
sd = fastsim.SimDrive(veh, cyc)
sd.walk()
print("Simulation complete.")


Simulation complete.


## Extracting Results

After `walk()`, the simulation results are stored inside the `SimDrive` object.
There are two primary ways to access them.

### `to_dataframe()`

Returns a tidy `pandas.DataFrame` with one row per time step and one column per
tracked quantity. Column names use dot-separated paths that mirror the object
hierarchy (`cyc.time_seconds`, `veh.history.speed_ach_meters_per_second`, etc.).

This is the most convenient format for plotting and analysis.


In [4]:
df = sd.to_dataframe()
print(f"{len(df)} time steps, {len(df.columns)} columns")
print("\nFirst few column names:")
print(df.columns.tolist()[:8])


1370 time steps, 62 columns

First few column names:
['veh.pt_type.Conv.fc.history.i', 'veh.pt_type.Conv.fc.history.pwr_out_max_watts', 'veh.pt_type.Conv.fc.history.pwr_prop_max_watts', 'veh.pt_type.Conv.fc.history.eff', 'veh.pt_type.Conv.fc.history.pwr_prop_watts', 'veh.pt_type.Conv.fc.history.energy_prop_joules', 'veh.pt_type.Conv.fc.history.pwr_aux_watts', 'veh.pt_type.Conv.fc.history.energy_aux_joules']


Plot achieved speed against the target cycle speed to confirm the vehicle
followed the trace:


In [5]:
import plotly.graph_objects as go

TARGET_COLOR = "#4D4D4D"
ACHIEVED_COLOR = "#0072B2"

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["cyc.time_seconds"],
    y=df["cyc.speed_meters_per_second"],
    name="Target",
    line={"dash": "dash", "width": 4, "color": TARGET_COLOR},
))
fig.add_trace(go.Scatter(
    x=df["cyc.time_seconds"],
    y=df["veh.history.speed_ach_meters_per_second"],
    name="Achieved",
    line={"color": ACHIEVED_COLOR},
))
fig.update_layout(
    xaxis_title="Time [s]",
    yaxis_title="Speed [m/s]",
    title="Target vs. Achieved Speed (UDDS)",
)
fig.show()


In [6]:
all(df["veh.history.cyc_met"])

True

FASTSim also has options for when the vehicle cannot meet the physical demands of the drive cycle. See [](trace-miss.ipynb) for more information.

### `to_pydict()`

`to_pydict(flatten=True)` returns a flat dictionary mapping dotted key paths to
scalar values (cumulative totals and final states). This is useful for
extracting summary metrics such as total fuel consumed or total distance driven.


In [7]:
sd_dict = sd.to_pydict(flatten=True)

fuel_joules = sd_dict["veh.pt_type.Conv.fc.state.energy_fuel_joules"]
dist_meters = sd_dict["veh.state.dist_meters"]

KWH_PER_GGE = 33.7
METERS_PER_MILE = 1609.34

fuel_kwh = fuel_joules / 3.6e6
miles = dist_meters / METERS_PER_MILE
mpg = miles / (fuel_kwh / KWH_PER_GGE)

print(f"Distance driven:  {miles:.2f} miles")
print(f"Fuel consumed:    {fuel_kwh:.3f} kWh  ({fuel_kwh / KWH_PER_GGE:.4f} gal)")
print(f"Fuel economy:     {mpg:.1f} mpgge")


Distance driven:  7.45 miles
Fuel consumed:    7.303 kWh  (0.2167 gal)
Fuel economy:     34.4 mpgge


## Controlling State History with `set_save_interval`

Predefined vehicles default to `save_interval = 1`, which records internal state at
every time step. This is required for `to_dataframe()` and per-step plots.

You can change this behavior:

| Value | Behavior |
|-------|----------|
| `1`   | Record every time step (default for predefined vehicles) |
| `N`   | Record every N-th time step |
| `None`| Disable history recording entirely (fastest; only final/cumulative state is available) |

Use `None` when you only need summary results (accumulated energy, distance, etc.)
and want to reduce memory usage and runtime:


In [8]:
veh_no_hist = fastsim.Vehicle.from_resource("2012_Ford_Fusion.yaml")
veh_no_hist.set_save_interval(None)  # disable per-step history

sd_no_hist = fastsim.SimDrive(veh_no_hist, cyc)
sd_no_hist.walk()

sd_dict_no_hist = sd_no_hist.to_pydict(flatten=True)
fuel_kwh_no_hist = sd_dict_no_hist["veh.pt_type.Conv.fc.state.energy_fuel_joules"] / 3.6e6
miles_no_hist = sd_dict_no_hist["veh.state.dist_meters"] / METERS_PER_MILE
mpg_no_hist = miles_no_hist / (fuel_kwh_no_hist / KWH_PER_GGE)
print(f"Fuel economy (no history): {mpg_no_hist:.1f} mpgge")


Fuel economy (no history): 34.4 mpgge


Without saving history, there will be no time-series simulation data to inspect.

In [9]:
sd_no_hist.to_dataframe().empty

True

## Running Multiple Simulations

To compare the same vehicle on different cycles, create new `SimDrive` objects
with the shared vehicle. This is efficient since the vehicle parameters are
only loaded once and reused.

To reset accumulated state and cumulative energy totals, call `sd.reset()` to
clear per-step history and totals without reloading the vehicle and cycle.

As an example, here the vehicle is simulated on both city (UDDS) and highway
(HWFET) cycles:


In [10]:
cyc_hwy = fastsim.Cycle.from_resource("hwfet.csv")

# Option 1: Create a new SimDrive with the highway cycle
sd_hwy = fastsim.SimDrive(veh, cyc_hwy)
sd_hwy.walk()

# Option 2: Reset and swap the cycle via pydict:
# sd.reset()
# sd_dict = sd.to_pydict()
# sd_dict["cyc"] = cyc_hwy.to_pydict()
# sd_hwy = fastsim.SimDrive.from_pydict(sd_dict)
# sd_hwy.walk()

sd_hwy_dict = sd_hwy.to_pydict(flatten=True)
fuel_kwh_hwy = sd_hwy_dict["veh.pt_type.Conv.fc.state.energy_fuel_joules"] / 3.6e6
miles_hwy = sd_hwy_dict["veh.state.dist_meters"] / METERS_PER_MILE
mpg_hwy = miles_hwy / (fuel_kwh_hwy / KWH_PER_GGE)

print(f"UDDS (city) fuel economy: {mpg:.1f} mpgge")
print(f"HWFET (highway) fuel economy: {mpg_hwy:.1f} mpgge")


UDDS (city) fuel economy: 34.4 mpgge
HWFET (highway) fuel economy: 47.0 mpgge


Note that these outputs are *unadjusted* fuel economy and do not accurately reflect the true label fuel economy. For information on matching label fuel economy with FASTSim, see [](label-fe.ipynb).